## **Pipeline di caricamento dati su SQL**

In questo notebook viene costruita la pipeline di caricamento dei dati puliti, nel database SQL. L’obiettivo è trasformare il dataset hotel_reviews_clean.csv nelle tabelle relazionali previste dallo schema (hotels, hotel_stats, reviewers, reviews), garantendo la coerenza tra chiavi primarie e chiavi esterne.

In questa prima fase vengono importate le librerie necessarie per la gestione dei dati e per la connessione al database. 

Successivamente viene caricato il dataset pulito, che costituisce la base di partenza della pipeline di inserimento.


## **Importazione delle librerie e caricamento del dataset pulito**

In [1]:
import pandas as p

from sqlalchemy import create_engine


In [2]:

url = r"C:\\Users\\anton\\Documents\\Lavoro\\hotel_reviews_clean.csv"
db= p.read_csv(url)

## **Connessione al database SQL**

In [3]:
engine = create_engine('mysql+mysqlconnector://root:@localhost/Hotel_review')

**Viene configurata la connessione al database SQL tramite SQLAlchemy**, così da consentire il trasferimento dei dataframe pandas nelle tabelle relazionali create in precedenza. Questo passaggio rende possibile il popolamento automatico del database direttamente da Python.

## **Popolamento della tabella Hotels**

In [17]:
hotels = db[['Hotel_Name', 'Hotel_Address', 'lat', 'lng']].drop_duplicates() 

hotels.columns = ['hotel_name', 'hotel_address', 'lat', 'lng']

La tabella **hotels** viene costruita selezionando le informazioni univoche relative a ciascun hotel, come nome, indirizzo e coordinate geografiche. La rimozione dei duplicati consente di ottenere una sola riga per struttura, evitando ripetizioni nella tabella anagrafica degli hotel.  


### Caricamento della tabella Hotels nel database

In [18]:
hotels.to_sql(

    name='hotels',

    con=engine,

    if_exists='append',

    index=False

)

1477

Dopo la preparazione del dataframe hotels, i dati vengono inseriti nel database tramite to_sql. In questo modo si popola la prima tabella dimensionale dello schema, che verrà successivamente utilizzata per recuperare gli identificativi necessari alle relazioni con le altre tabelle.  

### Primo tentativo di costruzione della tabella hotel_stats

In [19]:
hotel_stats = db[['Hotel_Name', 'Average_Score', 'Total_Number_of_Reviews', 'Additional_Number_of_Scoring']].drop_duplicates() 


hotel_stats.columns = ['hotel_id', 'average_score', 'total_number_of_reviews', 'additional_number_of_scoring']

La tabella **hotel_stats** viene derivata raggruppando il dataset pulito per hotel e mantenendo le informazioni statistiche associate alla struttura, come Average_Score, Total_Number_of_Reviews e Additional_Number_of_Scoring. Questa tabella contiene quindi le metriche sintetiche relative a ciascun hotel.

### Recupero degli hotel_id dalla tabella hotels

In [24]:
hotels = p.read_sql("SELECT hotel_id, hotel_name FROM hotels", con=engine)

### Prima merge tra hotel_stats e hotels

In [ ]:
hotel_stats = hotel_stats.merge(

    hotels,
    on="hotel_name",
    how="inner")

Le statistiche degli hotel vengono unite alla tabella hotels mediante una merge, così da associare a ciascuna struttura il corrispondente hotel_id. In questo modo la tabella hotel_stats risulta coerente con lo schema relazionale e pronta per l’inserimento nel database.  

### Controllo delle colonne dopo la merge

In [28]:
print(hotel_stats.columns)
print(hotels.columns)

Index(['hotel_name', 'average_score', 'total_number_of_reviews',
       'additional_number_of_scoring'],
      dtype='str')
Index(['hotel_id', 'hotel_name'], dtype='str')


### Costruzione definitiva della tabella hotel_stats

## **Popolamento tabella hotel_stats**

In [7]:
hotel_stats = (

    db.groupby(["Hotel_Name", "Hotel_Address"], as_index=False)

      .agg({

          "Average_Score": "first",

          "Total_Number_of_Reviews": "first",

          "Additional_Number_of_Scoring": "first"

      })

)

### Rinominazione delle colonne hotel_stats

In [8]:
hotel_stats.columns = [

    "hotel_name",

    "hotel_address",

    "average_score",

    "total_number_of_reviews",

    "additional_number_of_scoring"

]

Sezione delle colonne necessarie 

In [38]:
hotel_stats = hotel_stats[[

    "hotel_name",

    "average_score",

    "total_number_of_reviews",

    "additional_number_of_scoring"

]]

### Controllo della struttura di hotel_stats

In [10]:
print(hotel_stats.columns.tolist())

['hotel_name', 'hotel_address', 'average_score', 'total_number_of_reviews', 'additional_number_of_scoring']


### Recupero completo della tabella hotels

In [12]:
hotels_sql = p.read_sql(

    "SELECT hotel_id, hotel_name, hotel_address FROM hotels",

    con=engine

)

### Merge definitiva con hotel_id

In [13]:
hotel_stats = hotel_stats.merge(

    hotels_sql,

    on=["hotel_name", "hotel_address"],

    how="inner"

)

### Sezione finale delle colonne di hotel_stats

In [15]:
hotel_stats = hotel_stats[[

    "hotel_id",

    "average_score",

    "total_number_of_reviews",

    "additional_number_of_scoring"

]]

print(hotel_stats.columns.tolist())

print(hotel_stats.head())

['hotel_id', 'average_score', 'total_number_of_reviews', 'additional_number_of_scoring']
   hotel_id  average_score  total_number_of_reviews  \
0        78            8.7                      393   
1       120            7.7                      663   
2      1056            8.8                     4324   
3       530            9.6                      244   
4       549            9.4                       68   

   additional_number_of_scoring  
0                           101  
1                            69  
2                           391  
3                            66  
4                            27  


### Caricamento della tabella hotel_stats nel database

In [16]:
hotel_stats.to_sql(

    name="hotel_stats",

    con=engine,

    if_exists="append",

    index=False,

)

1477

## **Popolamento della tabella reviewers**

In [17]:
reviewers = db[[

    "Reviewer_Nationality",

    "Total_Number_of_Reviews_Reviewer_Has_Given"

]].drop_duplicates()

La tabella **reviewers** viene costruita selezionando le variabili Reviewer_Nationality e Total_Number_of_Reviews_Reviewer_Has_Given, poi eliminando i duplicati. In assenza di un identificativo univoco originale del recensore, questa combinazione viene utilizzata come criterio operativo per distinguere i profili dei recensori nel database.

### Rinominazione delle colonne di reviewers

In [18]:
reviewers.columns = [

    "reviewer_nationality",

    "total_number_of_reviews_reviewer_has_given"

]

Le colonne del dataframe reviewers vengono rinominate per renderle coerenti con la convenzione utilizzata nello schema SQL. Questo passaggio facilita la corrispondenza tra dataframe pandas e struttura del database, evitando ambiguità nei nomi delle variabili. 

### Controllo della struttura reviewers

In [19]:
print(reviewers.columns.tolist())

print(reviewers.head())

print(reviewers.shape)

['reviewer_nationality', 'total_number_of_reviews_reviewer_has_given']
  reviewer_nationality  total_number_of_reviews_reviewer_has_given
0              Russia                                            7
1             Ireland                                            7
2           Australia                                            9
3      United Kingdom                                            1
4         New Zealand                                            3
(6340, 2)


### Caricamento della tabella reviewers nel database

Il controllo sul dataframe reviewers mostra che il numero di combinazioni uniche ottenute è superiore al numero delle sole nazionalità. Questo accade perché il recensore viene identificato non solo tramite la nazionalità, ma anche tramite il numero totale di recensioni già rilasciate, generando così un insieme più dettagliato di profili.  

In [20]:
reviewers.to_sql(

    name="reviewers",

    con=engine,

    if_exists="append",

    index=False

)

6340

Il dataframe reviewers viene quindi inserito nel database tramite to_sql. In questo modo viene popolata la tabella dedicata ai recensori, che sarà poi richiamata nella costruzione della tabella reviews tramite reviewer_id.  

Avendo aggiunto una seconda variabile, il risultato è di 6340. Effettuando un controllo sul totale delle nazionalità il tot. è di 227.

In [21]:
print(db["Reviewer_Nationality"].nunique())

227


## **Popolamento della tabella reviews**

In [24]:
hotels_sql = p.read_sql(

    "SELECT hotel_id, hotel_name, hotel_address FROM hotels",

    con=engine

)

### Creazione delle variabili temporali

In [31]:
db["Review_Date"] = p.to_datetime(db["Review_Date"], errors="coerce")
db["review_year"] = db["Review_Date"].dt.year
db["review_month"] = db["Review_Date"].dt.month

### Selezione delle colonne per reviews

In [32]:
reviews = db[[

    "Hotel_Name",

    "Hotel_Address",

    "Reviewer_Nationality",

    "Total_Number_of_Reviews_Reviewer_Has_Given",

    "Review_Date",

    "review_year",

    "review_month",

    "Reviewer_Score",

    "Negative_Review",

    "Positive_Review",

    "Review_Total_Negative_Word_Counts",

    "Review_Total_Positive_Word_Counts",

    "Tags",

    "days_since_review",

    "stay_nights",

    "total_review_length" ]]

### **Rinominazione delle colonne**

In [33]:
reviews.columns = [
    "hotel_name",
    "hotel_address",
    "reviewer_nationality",
    "total_number_of_reviews_reviewer_has_given",
    "review_date",
    "review_year",
    "review_month",
    "reviewer_score",
    "negative_review",
    "positive_review",
    "review_total_negative_word_counts",
    "review_total_positive_word_counts",
    "tags",
    "days_since_review",
    "stay_nights",
    "total_review_length"
]

### Recupero delle chiavi esterne

In [34]:
hotels_sql = p.read_sql(

    "SELECT hotel_id, hotel_name, hotel_address FROM hotels",

    con=engine

)

reviewers_sql = p.read_sql(

    """

    SELECT reviewer_id, reviewer_nationality, total_number_of_reviews_reviewer_has_given

    FROM reviewers

    """,

    con=engine

)

### Collegamento di reviews con hotels e reviewers

In [35]:
reviews = reviews.merge(

    hotels_sql,

    on=["hotel_name", "hotel_address"],

    how="inner"

)

reviews = reviews.merge(

    reviewers_sql,

    on=["reviewer_nationality", "total_number_of_reviews_reviewer_has_given"],

    how="inner"

)

### Selezione finale delle colonne di reviews

In [36]:
reviews = reviews[[

    "hotel_id",

    "reviewer_id",

    "review_date",

    "review_year",

    "review_month",

    "reviewer_score",

    "negative_review",

    "positive_review",

    "review_total_negative_word_counts",

    "review_total_positive_word_counts",

    "tags",

    "days_since_review",

    "stay_nights",

    "total_review_length"

]]

Dopo il collegamento con le chiavi esterne, il dataframe reviews viene ridotto alle sole colonne necessarie per il database: identificativi, data, score, contenuti testuali, conteggi di parole, tag e variabili derivate. In questo modo la tabella finale contiene esclusivamente i campi previsti nello schema SQL.  

### Creazione della chiave primaria review_id

In [37]:
reviews.insert(0, "review_id", range(1, len(reviews) + 1))

 **Poiché la tabella reviews deve avere una chiave primaria univoca, viene aggiunta la colonna review_id, costruita come sequenza progressiva da 1 fino al numero totale delle recensioni. Questo garantisce l’identificazione univoca di ciascuna osservazione nella tabella finale.**

### Controllo finale della tabella reviews

In [38]:
print(reviews.columns.tolist())

print(reviews.head())

print(reviews.shape)

['review_id', 'hotel_id', 'reviewer_id', 'review_date', 'review_year', 'review_month', 'reviewer_score', 'negative_review', 'positive_review', 'review_total_negative_word_counts', 'review_total_positive_word_counts', 'tags', 'days_since_review', 'stay_nights', 'total_review_length']
   review_id  hotel_id  reviewer_id review_date  review_year  review_month  \
0          1         1            1  2017-08-03         2017             8   
1          2         1            2  2017-08-03         2017             8   
2          3         1            3  2017-07-31         2017             7   
3          4         1            4  2017-07-31         2017             7   
4          5         1            5  2017-07-24         2017             7   

   reviewer_score                                    negative_review  \
0             2.9   I am so angry that i made this post available...   
1             7.5                                        No Negative   
2             7.1   Rooms are n

### Caricamento della tabella reviews nel database

In [40]:
reviews.to_sql(

    name="reviews",

    con=engine,

    if_exists="append",

    index=False,

    chunksize=10000

)

511944

Una volta completata la struttura finale, il dataframe reviews viene inserito nel database con to_sql. Questo passaggio conclude la pipeline di caricamento, poiché popola la tabella centrale che collega hotel, recensori e contenuto delle recensioni.  

**Al termine della pipeline, il database risulta popolato con le quattro tabelle previste dallo schema: hotels, hotel_stats, reviewers e reviews.** La struttura ottenuta consente di svolgere correttamente le verifiche richieste e le successive analisi SQL su trend temporali, nazionalità dei recensori, performance degli hotel e caratteristiche delle recensioni.  
